In [ ]:
import pandas as pd
import numpy as np
import torch
import aiohttp
import asyncio
import faiss
from PIL import Image
Image.MAX_IMAGE_PIXELS = None
import io
from tqdm import tqdm
from transformers import CLIPProcessor, CLIPModel

In [ ]:
df = pd.read_csv("data/unsplash_lite/photos.csv", sep="\t")

clean_df = df[[
    "photo_id",
    "photo_image_url",
    "photo_description",
    "photographer_username"
]].dropna(subset=["photo_image_url"])

clean_df = clean_df.rename(columns={
    "photo_id": "id",
    "photo_image_url": "url",
    "photo_description": "caption",
    "photographer_username": "author"
})

clean_df.head()

In [ ]:
MODEL_NAME = "openai/clip-vit-large-patch14"

device = "cuda" if torch.cuda.is_available() else "cpu"

model = CLIPModel.from_pretrained(MODEL_NAME).to(device)
processor = CLIPProcessor.from_pretrained(MODEL_NAME)

model.eval()

In [ ]:
async def load_image_from_url(session, url):
    try:
        async with session.get(url, timeout=10) as r:
            if r.status != 200:
                return None

            data = await r.read()
            return Image.open(io.BytesIO(data)).convert("RGB").resize((224, 224))

    except:
        return None

In [ ]:
def embed_images(images):
    inputs = processor(images=images, return_tensors="pt", padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        features = model.get_image_features(**inputs)

    return features.cpu().numpy().astype("float32")

In [ ]:
async def run_batched(df, limit=1000, batch_size=16):

    embeddings = []
    ids = []
    
    connector = aiohttp.TCPConnector(limit=5)

    async with aiohttp.ClientSession(connector=connector) as session:

        batch_images = []
        batch_ids = []

        for _, row in tqdm(df.head(limit).iterrows(), total=limit):

            img = await load_image_from_url(session, row["url"])
            if img is None:
                continue

            batch_images.append(img)
            batch_ids.append(row["id"])

            # when batch is full → process
            if len(batch_images) == batch_size:

                vecs = embed_images(batch_images)

                embeddings.append(vecs)
                ids.extend(batch_ids)

                batch_images = []
                batch_ids = []

        # leftover batch
        if len(batch_images) > 0:
            vecs = embed_images(batch_images)
            embeddings.append(vecs)
            ids.extend(batch_ids)

    embeddings = np.concatenate(embeddings, axis=0).astype("float32")

    print("Final shape:", embeddings.shape)
    print("Total IDs:", len(ids))

    return ids, embeddings

In [ ]:
ids, embeddings = await run_batched(clean_df, 2000)

In [ ]:
embeddings = embeddings.astype("float32")
faiss.normalize_L2(embeddings)

d = embeddings.shape[1]
index = faiss.IndexFlatIP(d)
index.add(embeddings)

print("Indexed:", index.ntotal)

In [ ]:
import os
import pandas as pd

os.makedirs("data/unsplash", exist_ok=True)

# save FAISS index
faiss.write_index(index, "data/unsplash/index.faiss")

# build metadata (IMPORTANT: order must match embeddings)
meta = pd.DataFrame({
    "id": ids
})

# attach URLs in same order
id_to_url = dict(zip(clean_df["id"], clean_df["url"]))
meta["url"] = meta["id"].map(id_to_url)

meta.to_csv("data/unsplash/metadata.csv", index=False)

print("Saved index + metadata")